# Fires per Sentinel-2 grid, then MESMA

Run **in order**.

1. The report cell lists each grid: fire name and ignition date.
2. The map cell has Year / Grid dropdowns: blue = tile outline, orange = fire perimeter.
3. MESMA starts with **2015** (`YEARS_TO_RUN = [2015]`). After that finishes, change it to `[2016]` (then 2017…2020).
4. Outputs go to `new_data/2015-20_Mesma/{year}/{fire_name}/{tile}/`.


In [ ]:
from pathlib import Path
import json, os, re, csv, time, subprocess, sys, shutil
from collections import defaultdict

START_YEAR = 2015
END_YEAR = 2020

# Start with 2015. After that year finishes, change to [2016], then [2017], … [2020].
# Do not put all years here until 2015 is done.
YEARS_TO_RUN = [2015]
# Empty = every tile that had a fire that year. Example: ['11SLT']
RUN_TILES = []

# If you only have 2026-07-28 stacks, set 2026. None = match fire year to image year.
IMAGE_YEAR = None

SKIP_EXISTING = True
MAX_JOBS = 0          # 0 = all tiles for that year; 1 = trial
# True starts MESMA for YEARS_TO_RUN. False only prints the plan.
RUN_MESMA = True
CONDA_ENV = 'mesma'
CLASS_NAME = 'Type'

here = Path.cwd().resolve()
print('cwd', here)


In [ ]:
def first_existing(paths):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None

def find_new_data():
    checks = []
    for parent in [here, *here.parents]:
        checks.append(parent / 'new_data')
        checks.append(parent / 'SCE-CU' / 'sentinel2_data' / 'new_data')
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data'),
        Path('/home/jovyan/data-store/new_data'),
    ])
    return first_existing(checks) or Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data')

def find_fires_geojson(new_data):
    names = ('fires.geojson', 'fires_web.geojson', 'fires_2015_2026.geojson', 'fires_2015_2020.geojson')
    folders = (
        new_data / 'Time_since_fire' / 'by_year_2015_2026',
        new_data / 'Time_since_fire' / 'by_year_2015_2020',
        new_data / '2026-07-28' / 'atlas' / 'tsf_years',
        here / 'atlas' / 'tsf_years',
    )
    for folder in folders:
        for name in names:
            p = folder / name
            if p.exists():
                return p
    return None

def find_mesma_root():
    checks = []
    for parent in [here, *here.parents]:
        checks.append(parent / 'SCE-CU' / 'Mesma' / 'mesma')
        checks.append(parent / 'Mesma' / 'mesma')
        if (parent / 'mesma_main.py').exists():
            checks.append(parent)
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/Mesma/mesma'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/Mesma/mesma'),
        Path('/home/jovyan/data-store/SCE-CU/Mesma/mesma'),
        here,
    ])
    for p in checks:
        if p and (p / 'mesma_main.py').exists():
            return p
    return None

def find_library(mesma_root):
    names = ('38_output.sli', '39_output.sli')
    folders = []
    if mesma_root:
        folders += [mesma_root / 'internal_scripts' / 'spectral_library', mesma_root / 'spectral_library']
    folders.append(here / 'internal_scripts' / 'spectral_library')
    for folder in folders:
        for name in names:
            p = folder / name
            if p.exists():
                return p
    return None

NEW_DATA = find_new_data()
FIRES_PATH = find_fires_geojson(NEW_DATA)
MESMA_ROOT = find_mesma_root()
LIBRARY = find_library(MESMA_ROOT)
# SCE-CU/sentinel2_data/new_data/2015-20_Mesma/{year}/{fire_name}/{tile}/
OUT_DIR = (NEW_DATA / '2015-20_Mesma') if NEW_DATA else (here / '2015-20_Mesma')
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR = OUT_DIR

print('new_data :', NEW_DATA, 'OK' if NEW_DATA.exists() else 'MISSING')
print('fires    :', FIRES_PATH)
print('mesma    :', MESMA_ROOT)
print('library  :', LIBRARY)
print('output   :', OUT_DIR)


In [ ]:
%pip install -q shapely pandas matplotlib folium
from shapely.geometry import shape, Polygon, mapping
import pandas as pd
from IPython.display import display
print('shapely / pandas / folium OK')


In [ ]:
S2_KML_FOOTPRINTS = json.loads(r'''{"10SGB":{"ring":[[33.420926,-120.849393],[33.394855,-119.669903],[32.406061,-119.706744],[32.431169,-120.873216],[33.420926,-120.849393]],"center":[32.913494,-120.27156]},"11SKR":{"ring":[[32.496814,-120.192952],[32.521047,-119.02522],[31.53107,-119.003545],[31.507745,-120.158821],[32.496814,-120.192952]],"center":[32.014396,-119.598248]},"11SKS":{"ring":[[33.39762,-120.225533],[33.422698,-119.045911],[32.432875,-119.023246],[32.408724,-120.189844],[33.39762,-120.225533]],"center":[32.915711,-119.62439]},"11SKT":{"ring":[[34.298266,-120.259596],[34.324206,-119.067545],[33.33454,-119.043847],[33.309546,-120.222283],[34.298266,-120.259596]],"center":[33.816876,-119.651722]},"11SKU":{"ring":[[35.198208,-120.295184],[35.225028,-119.090147],[34.235522,-119.065374],[34.209667,-120.256178],[35.198208,-120.295184]],"center":[34.717347,-119.680279]},"11SKV":{"ring":[[36.098523,-120.332407],[36.126243,-119.11379],[35.136899,-119.087891],[35.110165,-120.291632],[36.098523,-120.332407]],"center":[35.618204,-119.710149]},"11SLA":{"ring":[[37.025276,-119.24838],[37.042247,-118.014303],[36.052541,-118.001452],[36.036168,-119.219917],[37.025276,-119.24838]],"center":[36.539207,-118.624916]},"11SLB":{"ring":[[37.925582,-119.275528],[37.943111,-118.02656],[36.953559,-118.013125],[36.936642,-119.245772],[37.925582,-119.275528]],"center":[37.439877,-118.644327]},"11SLR":{"ring":[[32.519328,-119.129284],[32.533687,-117.960535],[31.543236,-117.950248],[31.529415,-119.106497],[32.519328,-119.129284]],"center":[32.031551,-118.539766]},"11SLS":{"ring":[[33.420919,-119.151037],[33.435779,-117.970355],[32.445473,-117.959598],[32.431162,-119.127209],[33.420919,-119.151037]],"center":[32.93347,-118.555318]},"11SLT":{"ring":[[34.322366,-119.17378],[34.337737,-117.980622],[33.347578,-117.969375],[33.332767,-119.148867],[34.322366,-119.17378]],"center":[33.835252,-118.571578]},"11SLU":{"ring":[[35.223125,-119.197542],[35.239018,-117.99135],[34.249008,-117.979592],[34.233687,-119.171497],[35.223125,-119.197542]],"center":[34.736352,-118.588567]},"11SLV":{"ring":[[36.124276,-119.222396],[36.140702,-118.002571],[35.150843,-117.990279],[35.135002,-119.19517],[36.124276,-119.222396]],"center":[35.637852,-118.606338]},"11SMA":{"ring":[[37.04125,-118.124935],[37.046532,-116.890241],[36.056675,-116.891632],[36.051579,-118.110684],[37.04125,-118.124935]],"center":[36.549055,-117.507588]},"11SMB":{"ring":[[37.942081,-118.138529],[37.947537,-116.888914],[36.95783,-116.890368],[36.952565,-118.12363],[37.942081,-118.138529]],"center":[37.450051,-117.513722]},"11SMR":{"ring":[[32.532843,-118.065306],[32.537312,-116.89606],[31.546725,-116.897173],[31.542424,-118.053897],[32.532843,-118.065306]],"center":[32.039868,-117.480683]},"11SMS":{"ring":[[33.434905,-118.076197],[33.439531,-116.894997],[32.449086,-116.896161],[32.444632,-118.064267],[33.434905,-118.076197]],"center":[32.942081,-117.485597]},"11SMT":{"ring":[[34.336833,-118.087583],[34.341617,-116.893886],[33.351317,-116.895103],[33.346707,-118.07511],[34.336833,-118.087583]],"center":[33.844162,-117.490735]},"11SMU":{"ring":[[35.238084,-118.09948],[35.24303,-116.892725],[34.252876,-116.893998],[34.248107,-118.08644],[35.238084,-118.09948]],"center":[34.745569,-117.496103]},"11SMV":{"ring":[[36.139737,-118.111925],[36.144849,-116.89151],[35.154843,-116.892841],[35.149912,-118.098293],[36.139737,-118.111925]],"center":[35.647381,-117.501718]},"11SNA":{"ring":[[37.046583,-117.000225],[37.040161,-115.765562],[36.050528,-115.7812],[36.056724,-117.000222],[37.046583,-117.000225]],"center":[36.548556,-116.382893]},"11SNB":{"ring":[[37.94759,-117.000228],[37.940956,-115.750646],[36.951479,-115.766995],[36.957881,-117.000225],[37.94759,-117.000228]],"center":[37.449535,-116.375437]},"11SNR":{"ring":[[32.537355,-117.000213],[32.531922,-115.830993],[31.541537,-115.843511],[31.546767,-117.000211],[32.537355,-117.000213]],"center":[32.039446,-116.415603]},"11SNS":{"ring":[[33.439575,-117.000215],[33.433952,-115.819042],[32.443714,-115.832133],[32.449129,-117.000213],[33.439575,-117.000215]],"center":[32.941644,-116.409629]},"11SNT":{"ring":[[34.341663,-117.000217],[34.335847,-115.806548],[33.345757,-115.820235],[33.351361,-117.000215],[34.341663,-117.000217]],"center":[33.84371,-116.403383]},"11SNU":{"ring":[[35.243078,-117.00022],[35.237064,-115.793493],[34.247124,-115.807802],[34.252922,-117.000217],[35.243078,-117.00022]],"center":[34.745101,-116.396857]},"11SNV":{"ring":[[36.144898,-117.000222],[36.138683,-115.779838],[35.148896,-115.794796],[35.15489,-117.00022],[36.144898,-117.000222]],"center":[35.646897,-116.39003]},"11SPA":{"ring":[[37.041254,-115.875514],[37.023138,-114.641532],[36.034105,-114.671386],[36.051583,-115.88976],[37.041254,-115.875514]],"center":[36.53768,-115.265646]},"11SPB":{"ring":[[37.942085,-115.861926],[37.923373,-114.613058],[36.934511,-114.644268],[36.952569,-115.87682],[37.942085,-115.861926]],"center":[37.438298,-115.244939]},"11SPR":{"ring":[[32.532847,-115.93512],[32.517519,-114.766447],[31.527674,-114.790348],[31.542427,-115.946524],[32.532847,-115.93512]],"center":[32.03026,-115.356486]},"11SPS":{"ring":[[33.434909,-115.924234],[33.419047,-114.743632],[32.429359,-114.768624],[32.444635,-115.936159],[33.434909,-115.924234]],"center":[32.932134,-115.339895]},"11SPT":{"ring":[[34.336837,-115.912852],[34.320429,-114.719777],[33.330901,-114.745908],[33.346711,-115.92532],[34.336837,-115.912852]],"center":[33.833869,-115.322549]},"11SPU":{"ring":[[35.238088,-115.900959],[35.221123,-114.694855],[34.231757,-114.722171],[34.248111,-115.913994],[35.238088,-115.900959]],"center":[34.734922,-115.304424]},"11SPV":{"ring":[[36.139741,-115.888519],[36.122207,-114.668785],[35.133006,-114.697342],[35.149916,-115.902146],[36.139741,-115.888519]],"center":[35.636374,-115.285466]},"11SQA":{"ring":[[37.025284,-114.752069],[36.995511,-113.519414],[36.007451,-113.563414],[36.036176,-114.780526],[37.025284,-114.752069]],"center":[36.516368,-114.14997]},"11SQB":{"ring":[[37.925591,-114.724927],[37.894839,-113.477449],[36.906972,-113.523446],[36.93665,-114.754677],[37.925591,-114.724927]],"center":[37.416281,-114.116063]},"11SQR":{"ring":[[32.519335,-114.871141],[32.494142,-113.703535],[31.505174,-113.738769],[31.529422,-114.893924],[32.519335,-114.871141]],"center":[32.012254,-114.29873]},"11SQS":{"ring":[[33.420926,-114.849393],[33.394855,-113.669903],[32.406061,-113.706744],[32.431169,-114.873216],[33.420926,-114.849393]],"center":[32.913494,-114.27156]},"11SQT":{"ring":[[34.322373,-114.826654],[34.295406,-113.634741],[33.30679,-113.673258],[33.332775,-114.851562],[34.322373,-114.826654]],"center":[33.814582,-114.243152]},"11SQU":{"ring":[[35.223133,-114.802898],[35.195251,-113.598005],[34.206817,-113.63827],[34.233695,-114.828937],[35.223133,-114.802898]],"center":[34.714975,-114.213471]},"11SQV":{"ring":[[36.124284,-114.778048],[36.095467,-113.559581],[35.107218,-113.601672],[35.13501,-114.805269],[36.124284,-114.778048]],"center":[35.615751,-114.182425]}}''')

def tile_polygon(code):
    rec = S2_KML_FOOTPRINTS.get(code)
    if not rec:
        return None
    return Polygon([(lon, lat) for lat, lon in rec['ring']])

def tile_folder_name(code):
    return f'{code[:2]}_{code[2]}_{code[3:]}'

print(len(S2_KML_FOOTPRINTS), 'Sentinel-2 grids')


In [ ]:
from collections import defaultdict

if FIRES_PATH is None:
    raise FileNotFoundError('No fires.geojson yet. Run tsf_by_year_2015_2020.ipynb first.')

raw = json.loads(FIRES_PATH.read_text())
features = raw.get('features') or []

all_fires = []
for feat in features:
    props = feat.get('properties') or {}
    geom = feat.get('geometry')
    if not geom:
        continue
    try:
        year = int(props.get('burn_year'))
        poly = shape(geom)
    except (TypeError, ValueError, Exception):
        continue
    if year < START_YEAR or year > END_YEAR or poly.is_empty:
        continue
    name = (props.get('Incid_Name') or '').strip() or 'Unnamed fire'
    date = str(props.get('burn_date') or '').strip()
    all_fires.append({'name': name, 'date': date, 'year': year, 'geom': poly})

fire_rows = []
seen = set()
for code in S2_KML_FOOTPRINTS:
    tile = tile_polygon(code)
    if tile is None or tile.is_empty:
        continue
    for f in all_fires:
        if not tile.intersects(f['geom']):
            continue
        key = (code, f['name'], f['date'])
        if key in seen:
            continue
        seen.add(key)
        fire_rows.append({
            'grid': code,
            'folder': tile_folder_name(code),
            'fire_name': f['name'],
            'ignition_date': f['date'],
            'year': f['year'],
        })

# One MESMA job per (year, tile), with the fires that burned in that tile that year.
groups = defaultdict(list)
for r in fire_rows:
    groups[(r['year'], r['grid'])].append(r)

hits = []
for (year, grid), rows in sorted(groups.items()):
    rows = sorted(rows, key=lambda r: (r['ignition_date'] or '', r['fire_name']))
    hits.append({
        'year': year,
        'grid': grid,
        'folder': tile_folder_name(grid),
        'n_fires': len(rows),
        'fires': [{'name': r['fire_name'], 'date': r['ignition_date']} for r in rows],
        'fire_names': ', '.join(
            f"{r['fire_name']} ({r['ignition_date']})" if r['ignition_date'] else r['fire_name']
            for r in rows
        ),
    })

print(f'{len(all_fires)} fires → {len(hits)} year×tile jobs')
for year in range(START_YEAR, END_YEAR + 1):
    n = sum(1 for h in hits if h['year'] == year)
    print(f'  {year}: {n} tiles')


## Fires in each grid

One Sentinel-2 grid at a time: fire name and ignition date. CSV is written next to this notebook only.


In [ ]:
detail = pd.DataFrame(fire_rows, columns=['grid', 'fire_name', 'ignition_date', 'year'])
if detail.empty:
    raise RuntimeError('No fires overlapped any Sentinel-2 grid in this year range.')

detail = detail.sort_values(['grid', 'ignition_date', 'fire_name']).reset_index(drop=True)

pd.set_option('display.max_rows', 800)
pd.set_option('display.max_colwidth', 80)

print(f'{detail["grid"].nunique()} grids, {len(detail)} fires ({START_YEAR}–{END_YEAR})')
print()
for grid, grp in detail.groupby('grid', sort=True):
    print(f'{grid}   {len(grp)} fires')
    for _, row in grp.iterrows():
        print(f"  {row['fire_name']:<32}  ignition {row['ignition_date'] or '—'}")
    print()

print('CSV columns: grid, fire_name, ignition_date, year')
display(detail)

csv_path = OUT_DIR / 'fires_by_grid.csv'
detail.to_csv(csv_path, index=False)
print('Saved:', csv_path.resolve())

from IPython.display import FileLink
display(FileLink(str(csv_path), result_html_prefix='Download CSV: '))


## Grid + fire perimeters

Use the dropdowns: **Year**, then **Grid**. Blue = Sentinel-2 tile. Orange = fire perimeter for that grid.

In [ ]:
%pip install -q ipywidgets folium
import folium
from folium import Map, Polygon as FoliumPoly, GeoJson, GeoJsonTooltip, LayerControl
import ipywidgets as widgets
from IPython.display import display, clear_output

def ring_latlon(code):
    rec = S2_KML_FOOTPRINTS.get(code)
    if not rec:
        return None
    return [(lat, lon) for lat, lon in rec['ring']]

def grids_for_year(year):
    return ['All grids'] + sorted({h['grid'] for h in hits if h['year'] == year})

def map_selection(year, grid_choice):
    year_hits = [h for h in hits if h['year'] == year]
    year_fires = [f for f in all_fires if f['year'] == year]
    if grid_choice and grid_choice != 'All grids':
        want = grid_choice.replace('_', '').upper()
        year_hits = [h for h in year_hits if h['grid'].upper() == want]
        tile = tile_polygon(want)
        year_fires = [f for f in year_fires if tile is not None and tile.intersects(f['geom'])]
    bounds = []
    for h in year_hits:
        ring = ring_latlon(h['grid'])
        if ring:
            bounds.extend(ring)
    for f in year_fires:
        minx, miny, maxx, maxy = f['geom'].bounds
        bounds.extend([(miny, minx), (maxy, maxx)])
    if not bounds:
        print(f'Nothing to map for {year} {grid_choice}.')
        return None
    lat = sum(p[0] for p in bounds) / len(bounds)
    lon = sum(p[1] for p in bounds) / len(bounds)
    m = Map(location=[lat, lon], zoom_start=8, tiles=None)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
        attr='Tiles © Esri', name='Streets',
    ).add_to(m)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Tiles © Esri', name='Satellite',
    ).add_to(m)
    for h in year_hits:
        ring = ring_latlon(h['grid'])
        if not ring:
            continue
        FoliumPoly(
            locations=ring, color='#2450a0', weight=2.2, fill=True, fill_color='#2450a0',
            fill_opacity=0.05,
            popup=f"{h['grid']}<br>{h['n_fires']} fires<br>{h['fire_names']}",
            tooltip=h['grid'],
        ).add_to(m)
    for f in year_fires:
        GeoJson(
            {
                'type': 'Feature',
                'properties': {'name': f['name'], 'date': f['date'] or '—', 'year': f['year']},
                'geometry': mapping(f['geom']),
            },
            style_function=lambda _feat: {
                'color': '#7A2313', 'weight': 1.5, 'fillColor': '#ff9f00', 'fillOpacity': 0.45,
            },
            tooltip=GeoJsonTooltip(fields=['name', 'date'], aliases=['Fire', 'Ignition']),
        ).add_to(m)
    LayerControl(collapsed=True).add_to(m)
    m.fit_bounds([[min(p[0] for p in bounds), min(p[1] for p in bounds)],
                  [max(p[0] for p in bounds), max(p[1] for p in bounds)]])
    print(f"{year}  {grid_choice}: {len(year_hits)} grid(s), {len(year_fires)} fire perimeter(s)")
    for h in year_hits:
        print(f"  {h['grid']}  {h['n_fires']} fires  {h['fire_names']}")
    return m

years = sorted({h['year'] for h in hits}) or [START_YEAR]
year_dd = widgets.Dropdown(options=years, value=years[0], description='Year', layout=widgets.Layout(width='180px'))
grid_dd = widgets.Dropdown(options=grids_for_year(year_dd.value), description='Grid', layout=widgets.Layout(width='220px'))
map_out = widgets.Output()

def on_year_change(change):
    opts = grids_for_year(change['new'])
    grid_dd.options = opts
    if grid_dd.value not in opts:
        grid_dd.value = opts[0]

def redraw(*_):
    with map_out:
        clear_output(wait=True)
        m = map_selection(year_dd.value, grid_dd.value)
        if m is not None:
            display(m)

year_dd.observe(on_year_change, names='value')
year_dd.observe(redraw, names='value')
grid_dd.observe(redraw, names='value')
display(widgets.HBox([year_dd, grid_dd]), map_out)
redraw()

## Run MESMA for one year

Right now this runs **2015 only**. When 2015 is done, set `YEARS_TO_RUN = [2016]` and run this cell again.

Outputs:

`new_data/2015-20_Mesma/2015/Woolsey/11_S_LT/`

One folder per year, then one folder per fire, then the tile MESMA cube.


In [ ]:
IMAGE_NAME = re.compile(
    r'^(?P<z>\d{2})_(?P<band>[A-Z])_(?P<sq>[A-Z]{2})_(?P<y>\d{4})_(?P<m>\d{1,2})_(?P<d>\d{1,2})$'
)

def parse_tif(path: Path):
    m = IMAGE_NAME.match(path.stem.replace(' ', ''))
    if not m:
        return None
    return {
        'path': path,
        'tile': f"{m['z']}{m['band']}{m['sq']}",
        'year': int(m['y']),
        'date': f"{int(m['y']):04d}-{int(m['m']):02d}-{int(m['d']):02d}",
    }

def list_preprocessed_tifs(new_data: Path):
    found = {}
    roots = []
    if new_data and new_data.exists():
        roots.append(new_data)
    if MESMA_ROOT:
        roots.append(MESMA_ROOT / 'images')
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob('*'):
            if not p.is_file() or p.suffix.lower() not in {'.tif', '.tiff'}:
                continue
            if 'mesma' in p.name.lower() or '2015-20_Mesma' in p.parts or '2015-26_Mesma' in p.parts or 'Time_since_fire' in p.parts:
                continue
            meta = parse_tif(p)
            if meta:
                found[str(p)] = meta
    return list(found.values())

def pick_tif(tile, fire_year, tifs):
    same = [t for t in tifs if t['tile'] == tile]
    if not same:
        return None
    prefer = IMAGE_YEAR if IMAGE_YEAR else fire_year
    same.sort(key=lambda t: (0 if t['year'] == prefer else 1, abs(t['year'] - prefer), t['date']))
    return same[0]

def safe_fire_name(name, date=''):
    raw = (name or '').strip() or 'Unnamed_fire'
    raw = re.sub(r'[\\/:*?"<>|]+', '_', raw)
    raw = re.sub(r'\s+', '_', raw).strip('._')[:80] or 'Unnamed_fire'
    if raw.lower() in {'unnamed_fire', 'unnamed'} and date:
        raw = f'Unnamed_{date}'
    return raw

def fire_out_dir(year, fire_name, tile_folder, date='') -> Path:
    return OUT_DIR / str(year) / safe_fire_name(fire_name, date) / tile_folder

def job_fire_dirs(job) -> list:
    return [fire_out_dir(job['year'], f['name'], job['folder'], f.get('date') or '') for f in (job.get('fires') or [])]

def output_prefix(job, image: Path) -> Path:
    dirs = job_fire_dirs(job) or [OUT_DIR / str(job['year']) / job['folder']]
    dirs[0].mkdir(parents=True, exist_ok=True)
    return dirs[0] / image.stem

def has_mesma_output(job, image: Path) -> bool:
    for d in job_fire_dirs(job):
        if d.is_dir() and (any(d.glob(f'{image.stem}_mesma_*')) or any(d.glob('*_fractions'))):
            return True
    return False

def mirror_outputs(src_dir: Path, dest_dirs: list):
    files = [p for p in src_dir.iterdir() if p.is_file()]
    for dest in dest_dirs:
        if dest.resolve() == src_dir.resolve():
            continue
        dest.mkdir(parents=True, exist_ok=True)
        for f in files:
            target = dest / f.name
            if target.exists():
                continue
            try:
                os.link(f, target)
            except OSError:
                shutil.copy2(f, target)

def conda_mesma_cmd(image: Path, out_prefix: Path):
    return [
        'conda', 'run', '--no-capture-output', '-n', CONDA_ENV,
        'python', str(MESMA_ROOT / 'mesma_main.py'),
        str(LIBRARY), CLASS_NAME, str(image),
        '-o', str(out_prefix),
    ]

if MESMA_ROOT is None or LIBRARY is None:
    raise FileNotFoundError(
        'Clone SCE-CU so mesma_main.py and 38_output.sli are on the data store:\n'
        '  git clone https://github.com/CU-SCE/SCE-CU.git'
    )

tifs = list_preprocessed_tifs(NEW_DATA)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('MESMA output folder:', OUT_DIR)
print('Years this run:', YEARS_TO_RUN)
print(f'GeoTIFFs on disk: {len(tifs)}')

ok_all, failed_all = [], []
t0 = time.time()
for RUN_YEAR in YEARS_TO_RUN:
    year_hits = [h for h in hits if h['year'] == RUN_YEAR]
    if RUN_TILES:
        want = {t.replace('_', '').upper() for t in RUN_TILES}
        year_hits = [h for h in year_hits if h['grid'].upper() in want or h['folder'].upper() in {t.upper() for t in RUN_TILES}]
    jobs, missing = [], []
    for h in year_hits:
        img = pick_tif(h['grid'], h.get('year'), tifs)
        row = {**h, 'image': str(img['path']) if img else '', 'image_date': img['date'] if img else ''}
        (jobs if img else missing).append(row)
    todo, skipped = [], []
    for j in jobs:
        if SKIP_EXISTING and has_mesma_output(j, Path(j['image'])):
            skipped.append(j)
        else:
            todo.append(j)
    if MAX_JOBS and MAX_JOBS > 0:
        todo = todo[:MAX_JOBS]
    print('\n' + '#' * 72)
    print(f'YEAR {RUN_YEAR}  →  {OUT_DIR / str(RUN_YEAR)}')
    print(f'tiles with fires: {len(year_hits)}  queued: {len(todo)}  already done: {len(skipped)}  missing TIFF: {len(missing)}')
    if missing:
        print('No preprocessed image for:')
        for j in missing:
            print(f"  {j['grid']}  ({j['n_fires']} fires)  {j['fire_names']}")
    print('Plan:')
    for j in todo:
        print(f"  {j['grid']}  {j['n_fires']} fires  {j['fire_names']}")
    if not RUN_MESMA:
        print('RUN_MESMA is False — 2015 not started. Set RUN_MESMA = True and re-run this cell.')
        continue
    print('Starting', len(todo), 'tile(s) for', RUN_YEAR)
    for i, j in enumerate(todo, start=1):
        image = Path(j['image'])
        fire_dirs = job_fire_dirs(j)
        out_prefix = output_prefix(j, image)
        cmd = conda_mesma_cmd(image, out_prefix)
        print('\n' + '=' * 72)
        print(f"[{i}/{len(todo)}] {j['year']}  {j['grid']}  {j['n_fires']} fires")
        for f in j.get('fires') or []:
            print(f"  {f['name']}  ignition {f.get('date') or '—'}")
            print('   →', fire_out_dir(j['year'], f['name'], j['folder'], f.get('date') or ''))
        print(' '.join(cmd), flush=True)
        try:
            proc = subprocess.run(cmd, cwd=str(MESMA_ROOT), check=False)
            if proc.returncode == 0:
                mirror_outputs(out_prefix.parent, fire_dirs)
                ok_all.append(j)
                print('OK', image.name)
            else:
                failed_all.append((j, f'exit {proc.returncode}'))
                print('FAILED exit', proc.returncode)
        except Exception as exc:
            failed_all.append((j, str(exc)))
            print('FAILED', exc)

print('\n' + '#' * 72)
print(f"Done. success={len(ok_all)} failed={len(failed_all)}  {(time.time()-t0)/60:.1f} min")
print('Next: set YEARS_TO_RUN = [2016] and re-run this cell.')
for j, err in failed_all:
    print(f" - {j['year']} {j['grid']}: {err}")
